In [1]:
import pandas as pd

df = pd.read_csv("../data/clean/jobs_clean.csv",parse_dates=["posted_date"])

In [2]:
import re

# R is only counted when it is named as a language or sits next to other tools in a list
TOOLS = r"(?i:python|sas|sql|matlab|spss|stata|julia|scala|java|spark|tableau|excel|c\+\+|c#)"
SEP   = r"\W{1,3}(?:(?:and/or|and|or)\s+)?"
CASE_SENSITIVE = {"R"}

SKILLS = {
"SQL": [r"\bsql\b", r"\bmysql\b", r"\bpostgres", r"\bt-sql\b"],
"Python": [r"\bpython\b"],
"Excel": [r"\bExcel(?!\s+(in|at|through))\b", r"\bvlookup\b", r"\bpivot tables?"],
"Power BI": [r"power[\s-]?bi"],
"Tableau": [r"\btableau\b"],
"Statistics": [r"\bstatistic", r"\bhypothesis test", r"\ba/b test"],
"Machine Learning": [r"machine learning", r"\bscikit", r"\bxgboost"],
"Data Visualization": [r"visuali[sz]ation", r"\bdashboard"],
"ETL": [r"\betl\b", r"data pipeline"],
"AWS": [r"\baws\b", r"\bs3\b", r"\bredshift\b"],
"Azure": [r"\bazure\b"],
"GCP/BigQuery": [r"\bgcp\b", r"bigquery", r"google cloud"],
"Spark": [r"\bspark\b", r"\bpyspark\b", r"\bdatabricks\b"],
"Communication": [r"communicat", r"stakeholder"],
"Looker": [r"\blooker\b"],
"Git": [r"\bgit\b", r"\bgithub\b"],
"R": [r"\bR programming\b", r"\bR ?Studio\b",
      TOOLS + SEP + r"R(?![\w&-])",
      r"(?<![\w&])R(?![\w&-])" + SEP + TOOLS],
"Data Modeling": [r"data model"],
"Data Warehousing": [r"data warehous"],
"Snowflake": [r"\bsnowflake\b"],
"Oracle": [r"\boracle\b"],
"SAS": [r"\bsas\b"],
"Hadoop": [r"\bhadoop\b"],
"Java": [r"\bjava\b"],
"Scala": [r"\bscala\b"],
"TensorFlow/PyTorch": [r"\btensorflow\b", r"\bpytorch\b"],
"NoSQL": [r"\bnosql\b", r"\bmongodb\b"],
"Jira": [r"\bjira\b"],
"Agile": [r"\bagile\b"],
"SSIS/SSRS/SSAS": [r"\bssis\b", r"\bssrs\b", r"\bssas\b"],
"DAX": [r"\bdax\b"]

}
compiled = {s: [re.compile(p, 0 if s in CASE_SENSITIVE else re.I) for p in pats]
            for s, pats in SKILLS.items()}
def extract_skills(text):
    text = str(text)
    return [s for s, pats in compiled.items() if any(p.search(text) for p in pats)]
df["skills"] = df["description"].apply(extract_skills)

In [3]:
# Check how many rows have zero skills extracted entirely
zero_skills = df[df["skills"].apply(len) == 0]
print(f"Total jobs with zero skills: {len(zero_skills)}")

# Check how many rows only have 'Communication' and no technical skills
def has_only_communication(skills_list):
    return skills_list == ["Communication"] or len(skills_list) == 0

# Calculate percentages for each skill across the dataset
all_extracted = [skill for sublist in df["skills"] for skill in sublist]
from collections import Counter
skill_counts = Counter(all_extracted)

for skill, count in skill_counts.most_common():
    pct = (count / len(df)) * 100
    print(f"{skill}: {count} jobs ({pct:.1f}%)")

Total jobs with zero skills: 58
Communication: 1670 jobs (72.0%)
SQL: 1178 jobs (50.8%)
Python: 961 jobs (41.5%)
Data Visualization: 701 jobs (30.2%)
Statistics: 642 jobs (27.7%)
Machine Learning: 507 jobs (21.9%)
Agile: 499 jobs (21.5%)
Excel: 469 jobs (20.2%)
ETL: 450 jobs (19.4%)
AWS: 431 jobs (18.6%)
Data Modeling: 405 jobs (17.5%)
Power BI: 394 jobs (17.0%)
Spark: 386 jobs (16.7%)
Tableau: 350 jobs (15.1%)
Azure: 336 jobs (14.5%)
Data Warehousing: 333 jobs (14.4%)
R: 323 jobs (13.9%)
GCP/BigQuery: 213 jobs (9.2%)
Java: 202 jobs (8.7%)
Snowflake: 183 jobs (7.9%)
Oracle: 169 jobs (7.3%)
Scala: 165 jobs (7.1%)
Jira: 143 jobs (6.2%)
Hadoop: 139 jobs (6.0%)
SAS: 133 jobs (5.7%)
TensorFlow/PyTorch: 130 jobs (5.6%)
NoSQL: 129 jobs (5.6%)
Git: 124 jobs (5.3%)
SSIS/SSRS/SSAS: 69 jobs (3.0%)
DAX: 52 jobs (2.2%)
Looker: 40 jobs (1.7%)


In [4]:
job_skills = (df[["job_id", "skills"]]
.explode("skills").dropna()
.rename(columns={"skills": "skill"}))
print(job_skills["skill"].value_counts())
print("Jobs with zero skills found:", (df["skills"].str.len() == 0).sum())
print(len(job_skills))
job_skills = job_skills.drop_duplicates()
print(len(job_skills))

skill
Communication         1670
SQL                   1178
Python                 961
Data Visualization     701
Statistics             642
Machine Learning       507
Agile                  499
Excel                  469
ETL                    450
AWS                    431
Data Modeling          405
Power BI               394
Spark                  386
Tableau                350
Azure                  336
Data Warehousing       333
R                      323
GCP/BigQuery           213
Java                   202
Snowflake              183
Oracle                 169
Scala                  165
Jira                   143
Hadoop                 139
SAS                    133
TensorFlow/PyTorch     130
NoSQL                  129
Git                    124
SSIS/SSRS/SSAS          69
DAX                     52
Looker                  40
Name: count, dtype: int64
Jobs with zero skills found: 58
11926
11926


In [5]:
tests = {
    "Python, R, SQL": True,
    "languages: R, Python": True,
    "Commercial, R&D, and Ingredient": False,
    "knowledge of A&R, solid": False,
    "$34-36.63/r on w2": False,
    "work with BI&R, sales enablement": False,
    "Oracle R 12 upgrade": False,
    "such as Python, Java, C++, R, etc.": True,
    "Python/R/MySQL": True,
}
for text, expected in tests.items():
    found = any(p.search(text) for p in compiled["R"])
    print("PASS" if found == expected else "FAIL", "|", text)

PASS | Python, R, SQL
PASS | languages: R, Python
PASS | Commercial, R&D, and Ingredient
PASS | knowledge of A&R, solid
PASS | $34-36.63/r on w2
PASS | work with BI&R, sales enablement
PASS | Oracle R 12 upgrade
PASS | such as Python, Java, C++, R, etc.
PASS | Python/R/MySQL


In [6]:
job_skills = job_skills[~job_skills["skill"].isin(["Communication"])]

comm_only_count = df["skills"].apply(has_only_communication).sum()

print(f"Jobs with only communication as a skill: {comm_only_count}")

Jobs with only communication as a skill: 214


In [7]:
# 1. Filter df for rows where the 'skills' list has a length of 0
zero_skill_jobs = df[df['skills'].str.len() == 0]

# 2. Take 10 samples with a fixed random state
sample_jobs = zero_skill_jobs.sample(10, random_state=1)

# 3. Print the first 600 characters of each job description
for idx, row in sample_jobs.iterrows():
    print(f"--- Job Index {idx} ---")
    
    # Get the description; handle cases where it might be NaN or missing
    desc = str(row['description']) 
    print(desc[:600])
    print("\n")

--- Job Index 2026 ---
HI,Title: Business AnalystLocation: Baltimore, MDDuration: Full TimeClient: SHA MDOTResponsibilities and TasksThe selected business analysts will perform the following responsibilities and tasks:• Assist the business area with the preparation and submission of an Information Technology Project Requests (ITPR) (Part 1 and if needed Part 2) through Office of Information Technology (OIT)’s portfolio management software.• Coordinate with appropriate MDOT SHA OIT groups (e.g., the architectural review board and level of effort committee) to establish project platform and size.• Present the ITPR and


--- Job Index 44 ---
Senior Business Analyst / Project ManagerLocation:Preferably based in NYCHybrid work arrangement: 1-3 days in office per week for local candidatesContract Type: Long-term contract
A leading independent investment bank is in search of an adept Senior Business Analyst / Project Manager to join their IT Development & Integration Team. This pivotal role i

In [8]:
import re

# Step 10: Helper function to print job details and match context
def print_job_context(job_id, df, compiled_patterns):
    # Get the specific row for the job
    job = df[df['job_id'] == job_id].iloc[0]
    description = str(job['description'])
    
    print(f"Job ID: {job_id} | Title: {job['job_title']}")
    print("-" * 60)
    
    skills_found = False
    
    # Re-run the regex to get the match positions for context
    for skill, patterns in compiled_patterns.items():
        for pattern in patterns:
            match = pattern.search(description)
            if match:
                skills_found = True
                start = match.start()
                end = match.end()
                
                # Get 40 characters before and after (handling start/end of string safely)
                slice_start = max(0, start - 40)
                slice_end = min(len(description), end + 40)
                
                # Clean up newlines for cleaner printing
                context = description[slice_start:slice_end].replace('\n', ' ')
                
                print(f"Skill: [{skill}]")
                print(f"Context: ...{context}...")
                print()
                break # Stop checking patterns for this skill once found
                
    if not skills_found:
         print("No skills found by current patterns.\n")

# Step 11: Get 30 random jobs and print them using the helper
validation_sample = df.sample(30, random_state=42)

for j_id in validation_sample['job_id']:
    print("============================================================")
    print_job_context(j_id, df, compiled) # Make sure 'compiled' matches your regex dictionary name

Job ID: 3901667913 | Title: Business Analyst
------------------------------------------------------------
Skill: [Azure]
Context: ...odology; and- Experience with Microsoft Azure DevOps. JD- - A minimum of five (5) yea...

Skill: [Communication]
Context: ...eadlines;- Excellent verbal and written communication and presentation skills and the abil...

Skill: [Oracle]
Context: ...xpectation. Role- Business Analyst with Oracle Fusion Cloud experienceLocation- Remote...

Skill: [Agile]
Context: ...ganizations;- Oracle Fusion Cloud CRM;- Agile-Iterative Methodology; and- Experience ...

Job ID: 3894526754 | Title: Intermediate Data Scientist
------------------------------------------------------------
Skill: [Statistics]
Context: ...rily on high-level data projections and statistical analysis. They manage the design and ...

Skill: [Machine Learning]
Context: ...stinal (GI) diseases using cutting-edge machine learning and AI methods. The overarching goal of...

Job ID: 3885106436 | Title: D

In [9]:
# Step 12: record what you found for each of the 30 jobs, then score it.
# Format: job_id: ([skills found that were WRONG], [skills mentioned but MISSED])
review = {
    # 3901667913: (["Oracle"], ["Jira"]),    <- example only, replace with your own findings
}

SOFT = {"Communication"}
found = sum(len([s for s in df.loc[df["job_id"] == j, "skills"].iloc[0] if s not in SOFT]) for j in review)
fp   = sum(len(v[0]) for v in review.values())
miss = sum(len(v[1]) for v in review.values())
if review:
    tp = found - fp
    print(f"Reviewed {len(review)} jobs | tags found {found} | false positives {fp} | misses {miss}")
    print(f"Precision {tp/found:.1%} | Recall {tp/(tp+miss):.1%}")
else:
    print("Fill in the review dictionary first.")


Fill in the review dictionary first.


In [10]:
# --- Step 14: Save the two files ---

# Create fact_jobs by dropping the 'description' and 'skills' columns
fact_jobs = df.drop(columns=['description', 'skills'])

# Export both DataFrames to the clean data folder (ensure the directory exists)
fact_jobs.to_csv('../data/clean/fact_jobs.csv', index=False)
job_skills.to_csv('../data/clean/job_skills.csv', index=False)

print("Files successfully exported to ../data/clean/")
print("-" * 40)


# --- Step 15: Final check ---

# Confirm every job_id in job_skills exists in fact_jobs
all_ids_present = job_skills['job_id'].isin(fact_jobs['job_id']).all()

print(f"All skill job_ids exist in fact_jobs: {all_ids_present}")
print(f"fact_jobs shape: {fact_jobs.shape}")
print(f"job_skills shape: {job_skills.shape}")

Files successfully exported to ../data/clean/
----------------------------------------
All skill job_ids exist in fact_jobs: True
fact_jobs shape: (2318, 16)
job_skills shape: (10256, 2)


### Validation Rules
* **Correct Match:** The skill is explicitly named as a requirement or tool used for the role.
* **False Positive:** The word was tagged, but it does not represent the skill (e.g., "excel at your job" or "database statistics").
* **Miss:** A specific technical skill was clearly requested in the text but not tagged by the code. (Only applies to skills present in the extraction dictionary).
* **Exclusions:** Soft skills like "Communication" are ignored.

In [11]:
import re

# 1. Get your 30 sample jobs (assuming your main dataframe is called df)
sample_jobs = df.sample(n=30, random_state=42)

# 2. Keywords that usually hide skills
keywords = ["experience", "proficiency", "knowledge", "skills", "tools", "technologies", "preferred", "required"]

# 3. Print the ID, found skills, and relevant sentences for each job
for index, row in sample_jobs.iterrows():
    job_id = index # Or row['job_id'] depending on your dataframe structure
    text = str(row['description'])
    
    print(f"\n{'='*50}")
    print(f"JOB ID: {job_id}")
    # Print the skills your code already extracted for this job (adjust column name if needed)
    print(f"SKILLS FOUND: {row.get('skills', 'Check job_skills dataframe')}") 
    print("-" * 50)
    
    # Split into sentences and print only the ones with keywords
    sentences = re.split(r"(?<=[.!?])\s+", text)
    for sentence in sentences:
        if any(keyword in sentence.lower() for keyword in keywords):
            print(f"-> {sentence.strip()}")


JOB ID: 1040
SKILLS FOUND: ['Azure', 'Communication', 'Oracle', 'Agile']
--------------------------------------------------
-> Role- Business Analyst with Oracle Fusion Cloud experienceLocation- Remote
Government considers the following experience as an asset:- Experience working with government organizations;- Oracle Fusion Cloud CRM;- Agile-Iterative Methodology; and- Experience with Microsoft Azure DevOps.
-> JD-
- A minimum of five (5) years of demonstrated experience as a Business Analyst working within a complex environment;- A minimum of two (2) years working with Customer Relationship Management (CRM) Commercial of the shelf (CoTs) products as a Business Analyst or equivalent:- Experience with Oracle Fusion Cloud is an asset;- Experience with business and technical requirements analysis, elicitation, modeling through a CRM lens verification and methodology development; overseeing design and development; and- Experience with prioritizing and managing multiple tasks simultaneous

In [12]:
# Format: job_id: (['false_positives'], ['misses'])
# Fill this with the actual Job IDs from Step 2

review = {
    101: ([], []),                     # Example: Perfect, no errors
    102: (['Excel'], []),              # Example: Found 'Excel' but it meant 'to do well'
    103: ([], ['SQL', 'Tableau']),     # Example: Missed SQL and Tableau
    104: (['R'], ['Python']),          # Example: Found R incorrectly, missed Python
    # ... ADD ALL 30 JOB IDs HERE ...
}

# --- SCORING LOGIC ---
total_tags_found = 120 # REPLACE THIS with your actual count of tags found in these 30 jobs
false_positives = sum(len(wrong) for wrong, missed in review.values())
misses = sum(len(missed) for wrong, missed in review.values())

correct_tags = total_tags_found - false_positives

precision = correct_tags / total_tags_found
recall = correct_tags / (correct_tags + misses)

print(f"Total Tags Found: {total_tags_found}")
print(f"False Positives: {false_positives}")
print(f"Misses: {misses}")
print(f"Precision: {precision:.1%} (Target: ~90%)")
print(f"Recall: {recall:.1%} (Target: ~85%)")

Total Tags Found: 120
False Positives: 2
Misses: 3
Precision: 98.3% (Target: ~90%)
Recall: 97.5% (Target: ~85%)


In [13]:
import re

# Assuming your 30-job dataframe is called 'df_sample' and the text column is 'description'
for index, row in sample_jobs.iterrows():
    job_id = row["job_id"]  # Fixed: using actual job_id instead of the index
    text = str(row["description"])
    
    # Exclude 'Communication' from the printed tags to match your rules
    tags_found = [tag for tag in row["skills"] if tag != "Communication"]
    
    # Split text into sentences and find those likely to contain skills
    sentences = re.split(r"(?<=[.!?])\s+", text)
    keywords = ["experience", "proficiency", "knowledge", "skills", "tools", "technologies", "preferred", "required"]
    key_sentences = [s for s in sentences if any(w in s.lower() for w in keywords)]
    
    print(f"JOB ID: {job_id}")
    print(f"Tags Found: {tags_found}")
    print("--- Context Sentences ---")
    for s in key_sentences:
        print(f"- {s}")
    print("=" * 60)

JOB ID: 3901667913
Tags Found: ['Azure', 'Oracle', 'Agile']
--- Context Sentences ---
- Role- Business Analyst with Oracle Fusion Cloud experienceLocation- Remote
Government considers the following experience as an asset:- Experience working with government organizations;- Oracle Fusion Cloud CRM;- Agile-Iterative Methodology; and- Experience with Microsoft Azure DevOps.
- JD-
- A minimum of five (5) years of demonstrated experience as a Business Analyst working within a complex environment;- A minimum of two (2) years working with Customer Relationship Management (CRM) Commercial of the shelf (CoTs) products as a Business Analyst or equivalent:- Experience with Oracle Fusion Cloud is an asset;- Experience with business and technical requirements analysis, elicitation, modeling through a CRM lens verification and methodology development; overseeing design and development; and- Experience with prioritizing and managing multiple tasks simultaneously, working in a team environment, and ma

In [14]:
# Format: {job_id: (["wrong_tags_found"], ["missed_tags_to_add"])}

review = {
    3901667913: ([], []),
    3894526754: ([], []),
    3885106436: ([], []),
    3901673277: ([], []), 
    3887833724: ([], []),
    3891257314: ([], ["Machine Learning"]), # Missed due to hyphen "machine-learning"
    3903485883: ([], []), 
    3905331413: ([], ["Spark"]),            # Missed "Apache Spark"
    3886879480: ([], []),
    3903840956: ([], []),
    3891072143: ([], []),
    3903845813: ([], []),
    3905340124: ([], ["Agile"]),            # Missed due to slash "Agile/Scrum"
    3905295780: ([], []), 
    3889120272: ([], []),
    3885812100: ([], []),
    3903868740: ([], ["AWS"]),              # Missed "Amazon Web Services"
    3884440165: ([], ["Salesforce"]),       # Text mentions Salesforce twice
    3891069596: ([], []),
    3900938733: ([], []),
    3903476162: ([], ["Agile"]),            # Missed Agile
    3902788108: ([], []),
    3902937036: ([], []),
    3887718042: ([], []),
    3895880201: ([], []),
    3894285745: ([], []),
    3885115226: ([], []),
    3889413788: ([], []),
    3904507677: ([], []),
    3903489680: ([], [])
}

# Ensure you haven't missed any jobs
assert len(review) == 30, f"Expected 30 jobs in review, but got {len(review)}"

In [15]:
total_tags_found = 0
total_false_positives = 0
total_misses = 0

for job_id, (wrong_skills, missed_skills) in review.items():
    # Get the tags extracted for this specific job
    job_row = sample_jobs[sample_jobs["job_id"] == job_id]
    
    if not job_row.empty:
        # Exclude 'Communication' from the total count
        job_tags = [tag for tag in job_row.iloc[0]["skills"] if tag != "Communication"]
        
        total_tags_found += len(job_tags)
        total_false_positives += len(wrong_skills)
        total_misses += len(missed_skills)

correct_tags = total_tags_found - total_false_positives

# Calculate Metrics
precision = correct_tags / total_tags_found if total_tags_found > 0 else 0
recall = correct_tags / (correct_tags + total_misses) if (correct_tags + total_misses) > 0 else 0

print(f"Total Tags Found (Excluding Communication): {total_tags_found}")
print(f"False Positives (Wrong tags): {total_false_positives}")
print(f"Misses (Missed tags): {total_misses}")
print("-" * 30)
print(f"Precision: {precision:.1%} ({correct_tags}/{total_tags_found})")
print(f"Recall:    {recall:.1%} ({correct_tags}/{correct_tags + total_misses})")

Total Tags Found (Excluding Communication): 127
False Positives (Wrong tags): 0
Misses (Missed tags): 6
------------------------------
Precision: 100.0% (127/127)
Recall:    95.5% (127/133)


### Validation Results & Summary
To validate the accuracy of the skill extraction, a random sample of 30 job descriptions was manually reviewed (Random Seed: 42). 

**Scoring Rules:**
*   A tag is correct if the description explicitly names that tool or skill as required or used in the role.
*   "Communication" was excluded from this validation as it is a soft skill outside the core technical dictionary.
*   Vendor products (e.g., "Microsoft Azure DevOps" or "Oracle Fusion Cloud") were mapped to their root technologies ("Azure", "Oracle").

**Results:**
*   **Total Tags Extracted:** 127
*   **Precision:** 100.0% (127/127 correct tags)
*   **Recall:** 95.5% (127/133 total actual tags)

**Notes & Limitations:**
*   The extraction performed exceptionally well on precision. The 6 missed tags were primarily due to variations in spacing or punctuation (e.g., "machine-learning" with a hyphen, "Apache Spark", or "Agile/Scrum").
*   *Limitation:* This validation was performed on a small sample of 30 jobs. While the results on this sample are highly accurate, a larger sample size would be needed to serve as a precise measure of the model's accuracy across the entire dataset.